# Power laws: logarithmic coordinates and elasticity

An application of Alpha Calculus with explicit fitting, interpretation
and assumptions.

This notebook is self-contained. It assumes familiarity with logarithms,
elementary derivatives and basic Python. By the end, you should be able
to interpret a power law as a straight line in log-log coordinates and
identify what changes when the data are transformed.

1. Define dimensionless ratios and transport a positive function.
2. Fit a line and interpret its slope as elasticity.
3. Separate multiplicative and additive error and their fitting criteria.
4. Distinguish the mean from the median when returning from the log scale.

Run all cells in order in a fresh runtime. Only the standard library is used:
no other notebook, book, network connection, private data or Drive access is
required. Tables are exported as CSV files to
`results/power_laws_and_elasticity/`.


In [1]:
import csv
import json
import math
import platform
from pathlib import Path

RESULTS = Path("results") / "power_laws_and_elasticity"
RESULTS.mkdir(parents=True, exist_ok=True)

print("Python:", platform.python_version())
print("Float precision: approximately 16 significant digits.")

Python: 3.14.3
Float precision: approximately 16 significant digits.


In [2]:
def positive_log(value):
    """Coordinate of a positive, finite, dimensionless quantity."""
    if not math.isfinite(value) or value <= 0:
        raise ValueError("The quantity must be positive and finite.")
    return math.log(value)

In [3]:
def print_table(
        title,
        header,
        rows,
    ):
    """Print a small table with aligned columns."""
    data = [
        [str(value) for value in row]
        for row in [header, *rows]
        ]
    widths = [
        max(map(len, column))
        for column in zip(*data)
        ]

    print(title)
    for index, row in enumerate(data):
        print(
                " | ".join(
                        value.ljust(width)
                        for value, width in zip(row, widths)
                    )
            )
        if index == 0:
            print("-+-".join("-" * width for width in widths))


def export_table(
        name,
        header,
        rows,
    ):
    """Save table data as UTF-8 CSV."""
    with (RESULTS / name).open(
            "w",
            newline="",
            encoding="utf-8",
        ) as file:
        writer = csv.writer(file)
        writer.writerow(header)
        writer.writerows(rows)


## 1. Application: recognising a power law and interpreting its elasticity

### Situation, recognition cues and assumptions

A positive response may follow a scaling law. If multiplying the input by
$q$ multiplies the response by $q^b$, a power law is a candidate model.
An apparently straight relationship in log-log coordinates can guide
exploration; it does not establish the model or identify its noise.

Choose positive reference quantities $x_0,y_0$ with their units, and work
with dimensionless ratios:

$$X=x/x_0>0,\qquad Y=y/y_0>0,\qquad Y=A X^b,\quad A>0.$$

Logarithms therefore apply to ratios, rather than quantities with units.
The references remain fixed during the analysis. Changing them consistently
changes the intercept, but not the elasticity $b$.

### Step 1. Transporting the function

For $f(X)=A X^b$ and $\alpha=\exp$:

$$F(t)=\log(f(e^t))=\log A+bt.$$

The model is a straight line in coordinates $t=\log X$, $v=\log Y$.
The transported derivative is $D_\alpha f=e^b$; **its logarithm**, $b$,
is the elasticity. $D_\alpha f$ must not be confused with the ordinary
derivative $f'$, or described as being directly equal to $b$.

The direction $F=\alpha^{-1}\circ f\circ\alpha$ is derived from
Definition 4.1.1 and checked against Example 5.1.2. The intervening passage
contains local composition errors: see the source scope at the end.
For this case, the derivation above is direct.


In [4]:
A = 3.0
b = 1.5
X = [0.5, 1.0, 2.0, 4.0, 8.0]
Y = [A * value ** b for value in X]
t = [math.log(value) for value in X]
v = [math.log(value) for value in Y]

power_rows = [
    [f"{x:.3f}", f"{y:.6f}", f"{tx:.6f}", f"{vy:.6f}"]
    for x, y, tx, vy in zip(X, Y, t, v)
    ]
print_table(
        "From curve to straight line",
        ["X", "Y", "log(X)", "log(Y)"],
        power_rows,
    )
export_table(
        "power_coordinates.csv",
        ["X", "Y", "log_X", "log_Y"],
        power_rows,
    )

for tx, vy in zip(t, v):
    assert math.isclose(vy, math.log(A) + b * tx, abs_tol=1e-12)

From curve to straight line
X     | Y         | log(X)    | log(Y)  
------+-----------+-----------+---------
0.500 | 1.060660  | -0.693147 | 0.058892
1.000 | 3.000000  | 0.000000  | 1.098612
2.000 | 8.485281  | 0.693147  | 2.138333
4.000 | 24.000000 | 1.386294  | 3.178054
8.000 | 67.882251 | 2.079442  | 4.217775


### Step 2. Fitting the line and checking the back-transformation

The least-squares slope in logarithmic coordinates is

$$\hat b=\frac{\sum_i(t_i-\bar t)(v_i-\bar v)}
                       {\sum_i(t_i-\bar t)^2},\qquad
\hat a=\bar v-\hat b\bar t,\quad \hat A=e^{\hat a}.$$

This is conventional log-log fitting. The Alpha interpretation organises
the coordinates and their meaning; it does not introduce another estimator.
The sums cost $O(n)$, as does the identical conventional baseline.


In [5]:
def fit_line(abscissae, ordinates):
    """Fit an intercept and slope to finite, non-degenerate data."""
    if len(abscissae) != len(ordinates) or len(abscissae) < 2:
        raise ValueError(
                "Lists must have equal lengths and contain at least two values."
                )
    if not all(math.isfinite(value) for value in [*abscissae, *ordinates]):
        raise ValueError("Fit data must be finite.")
    mean_x = math.fsum(abscissae) / len(abscissae)
    mean_y = math.fsum(ordinates) / len(ordinates)
    x_differences = [value - mean_x for value in abscissae]
    denominator = math.fsum(value * value for value in x_differences)
    if denominator == 0.0:
        raise ValueError("The input must vary to identify the slope.")
    numerator = math.fsum(
            difference * (value - mean_y)
            for difference, value in zip(x_differences, ordinates)
            )
    slope = numerator / denominator
    return mean_y - slope * mean_x, slope

In [6]:
a_estimated, b_estimated = fit_line(t, v)
A_estimated = math.exp(a_estimated)
print(f"Estimated A = {A_estimated:.9f}; estimated b = {b_estimated:.9f}")
print(f"Alpha derivative = exp(b) = {math.exp(b_estimated):.9f}")

assert math.isclose(A_estimated, A, rel_tol=1e-12)
assert math.isclose(b_estimated, b, abs_tol=1e-12)

# Another input reference: X_new = x / (10*x0).
t_other_reference = [math.log(value / 10.0) for value in X]
a_other, b_other = fit_line(t_other_reference, v)
assert math.isclose(b_other, b, abs_tol=1e-12)
assert math.isclose(
        a_other,
        math.log(A) + b * math.log(10.0),
        abs_tol=1e-12,
    )
print("Changing the reference preserves b and shifts the intercept.")

Estimated A = 3.000000000; estimated b = 1.500000000
Alpha derivative = exp(b) = 4.481689070
Changing the reference preserves b and shifts the intercept.


### Step 3. Turning the slope into an explanation

The ordinary elasticity of this function is

$$\frac{Xf'(X)}{f(X)}=b.$$

A small 1% increase in the input produces an approximately $b$% increase
in the response. For a finite change by a factor $q$, the exact ratio is
$Y'/Y=q^b$. The percentage approximation is not an equality for every
change. This interpretation can guide a scaling hypothesis or calibration;
the data and mechanism must justify its use.


In [7]:
change_factor = 1.01
exact_change = change_factor ** b - 1.0
linear_change = b * (change_factor - 1.0)
secant_elasticity = math.log(change_factor ** b) / math.log(change_factor)

print_table(
        "Input +1% with b = 1.5",
        ["Quantity", "Value"],
        [
            ["Response: exact change", f"{100 * exact_change:.6f}%"],
            ["Local approximation", f"{100 * linear_change:.6f}%"],
            ["Exact logarithmic ratio", f"{secant_elasticity:.9f}"],
            ],
        )
assert math.isclose(secant_elasticity, b, rel_tol=1e-12)

Input +1% with b = 1.5
Quantity                | Value      
------------------------+------------
Response: exact change  | 1.503744%  
Local approximation     | 1.500000%  
Exact logarithmic ratio | 1.500000000


### Step 4. Adding errors requires a modelling decision

With **multiplicative** error,

$$Y_i=A X_i^b e^{\varepsilon_i}
\quad\Longrightarrow\quad
\log Y_i=\log A+b\log X_i+\varepsilon_i.$$

Interpreting log-OLS as estimation under normal noise assumes conditional
errors that are centred, normal and of constant variance on the log scale,
in addition to the design conditions. The algebraic identity does not impose
those assumptions. The following demonstration uses **chosen, fixed** errors,
without claiming that they form a normal sample.


In [8]:
t_demo = [-2.0, -1.0, 0.0, 1.0, 2.0]
chosen_errors = [0.15, -0.10, 0.05, 0.20, -0.05]
X_demo = [math.exp(value) for value in t_demo]
Y_demo = [
    A * x ** b * math.exp(error)
    for x, error in zip(X_demo, chosen_errors)
    ]
v_demo = [math.log(value) for value in Y_demo]
a_demo, b_demo = fit_line(t_demo, v_demo)

fit_rows = [
    [
        f"{x:.4f}",
        f"{y:.5f}",
        f"{error:+.2f}",
        f"{math.exp(a_demo) * x ** b_demo:.5f}",
        ]
    for x, y, error in zip(X_demo, Y_demo, chosen_errors)
    ]
print_table(
        "Demonstration with fixed errors",
        ["X", "Observed Y", "Log error", "Fitted curve"],
        fit_rows,
        )
export_table(
        "power_fit.csv",
        ["X", "Y", "log_error", "fitted_curve"],
        fit_rows,
        )
print(f"Fitted A = {math.exp(a_demo):.6f}; fitted b = {b_demo:.6f}")
assert math.isclose(a_demo, math.log(A) + 0.05, abs_tol=1e-12)
assert math.isclose(b_demo, 1.49, abs_tol=1e-12)

Demonstration with fixed errors
X      | Observed Y | Log error | Fitted curve
-------+------------+-----------+-------------
0.1353 | 0.17353    | +0.15     | 0.16019     
0.3679 | 0.60569    | -0.10     | 0.71078     
1.0000 | 3.15381    | +0.05     | 3.15381     
2.7183 | 16.42184   | +0.20     | 13.99377    
7.3891 | 57.31786   | -0.05     | 62.09170    
Fitted A = 3.153813; fitted b = 1.490000


### Limitation 1. Taking logarithms does not preserve the error criterion

Log-OLS minimises $\sum_i[\log Y_i-a-b\log X_i]^2$.
Fitting on the original scale minimises $\sum_i[Y_i-A X_i^b]^2$.
These address different questions.

Even with $b$ fixed and observations only at $X=1$, for $Y=[1,2,9]$ the
original-scale fit is the arithmetic mean; the logarithmic fit transforms
back to the geometric mean. Each minimises its own criterion. Neither can
be declared better by comparing errors measured on different scales.


In [9]:
observations = [1.0, 2.0, 9.0]
original_fit = math.fsum(observations) / len(observations)
mean_log = math.fsum(math.log(y) for y in observations) / len(observations)
log_fit = math.exp(mean_log)

criterion_rows = []
for name, fit in [
        ("Original LS", original_fit),
        ("Log LS", log_fit),
    ]:
    sse_original = math.fsum((y - fit) ** 2 for y in observations)
    sse_log = math.fsum(
            (math.log(y) - math.log(fit)) ** 2
            for y in observations
        )
    criterion_rows.append(
            [
                name,
                f"{fit:.6f}",
                f"{sse_original:.6f}",
                f"{sse_log:.6f}",
                ]
            )

print_table(
        "Same data, different objective",
        ["Fit", "Prediction", "Original SSE", "Log SSE"],
        criterion_rows,
        )
export_table(
        "power_criteria.csv",
        ["fit", "prediction", "sse_original", "sse_log"],
        criterion_rows,
        )
assert original_fit == 4.0
assert math.isclose(log_fit, 18.0 ** (1.0 / 3.0), rel_tol=1e-12)

Same data, different objective
Fit         | Prediction | Original SSE | Log SSE 
------------+------------+--------------+---------
Original LS | 4.000000   | 38.000000    | 3.059873
Log LS      | 2.620741   | 43.707063    | 2.523499


### Limitation 2. Applying `exp` does not automatically recover the mean

If $\varepsilon\mid X\sim N(0,\sigma^2)$, then

$$\operatorname{Median}(Y\mid X)=A X^b,\qquad
E[Y\mid X]=A X^b e^{\sigma^2/2}.$$

The mean and median agree only when the variance is zero.
This factor is valid under the stated lognormal model; estimating its
parameters adds uncertainty. It is not a universal correction for every
distribution or for variance that changes with $X$.


In [10]:
mean_rows = [
    [f"{sigma:.1f}", f"{A:.6f}", f"{A * math.exp(sigma ** 2 / 2):.6f}"]
    for sigma in [0.0, 0.3, 0.6, 1.0]
    ]
print_table(
        "At X = 1, A = 3: lognormal model",
        ["sigma", "Median", "Mean"],
        mean_rows,
    )
export_table(
        "power_mean_median.csv",
        ["sigma", "median", "mean"],
        mean_rows,
    )
assert math.exp(0.6 ** 2 / 2) > 1.0

At X = 1, A = 3: lognormal model
sigma | Median   | Mean    
------+----------+---------
0.0   | 3.000000 | 3.000000
0.3   | 3.000000 | 3.138084
0.6   | 3.000000 | 3.591652
1.0   | 3.000000 | 4.946164


### Limitation 3. Additive error can leave the domain

If the model were $Y=A X^b+\eta$, rewriting it would not produce the
multiplicative model. Moreover, zero or negative observations prevent the
use of these logarithms. The following check represents $X=1$, $A=3$ and
additive errors $[-4,0,4]$.

A constant is not automatically added to repair the domain:
that decision changes the function and its interpretation.


In [11]:
additive_responses = [A + error for error in [-4.0, 0.0, 4.0]]
print("Responses with additive error:", additive_responses)

try:
    [positive_log(y) for y in additive_responses]
except ValueError as error:
    print("Domain correctly rejected:", error)
else:
    raise AssertionError("The negative response should have been rejected.")

Responses with additive error: [-1.0, 3.0, 7.0]
Domain correctly rejected: The quantity must be positive and finite.


### Exercise

A curve is $Y=2X^{0.5}$. What happens when $X$ doubles? What is its
elasticity? What is its Alpha derivative for $\alpha=\exp$?

**Hint:** distinguish the exact factor $2^{0.5}$, the exponent $0.5$ and
$e^{0.5}$. Then explain why this deterministic question does not require
justification of a noise model.


In [12]:
b_exercise = 0.5
print(f"Factor when doubling X: {2.0 ** b_exercise:.9f}")
print(f"Elasticity: {b_exercise}")
print(f"Alpha derivative: {math.exp(b_exercise):.9f}")
assert math.isclose(2.0 ** b_exercise, math.sqrt(2.0), rel_tol=1e-12)

Factor when doubling X: 1.414213562
Elasticity: 0.5
Alpha derivative: 1.648721271


## 2. What has been checked and what needs further evidence

Changing to logarithmic coordinates turns a positive power law into a
straight line. Its slope is the constant elasticity: it explains relative
changes and helps recognise scaling or calibration hypotheses.
The Alpha derivative is $e^b$, and its logarithm is $b$.

The fit is conventional log-log regression. The Alpha interpretation
organises the coordinates and their meaning without adding a new estimator
or an exclusive advantage. The demonstration does not establish an empirical
law: the data are chosen. Transforming the response changes the error
criterion and requires checking the noise model and the prediction to be reported.

**Proposed extension:** choose public data and justify the mechanism,
reference units and error model before validating the fit and inference.

## Sources and scope

- Georgiev and Aydin, *Alpha Calculus*, 1st ed., 2026,
  [DOI 10.1201/9781003750734](https://doi.org/10.1201/9781003750734).
  Passages checked: pp. **1–2** (generator), **128** (Definition 4.1.1),
  **169–170** (equations 5.1–5.2 and Example 5.1.2). These correspond to
  PDF pp. 10–11, 137 and 178–179. Local omissions and a reversed composition
  on pp. 169–170 have been documented by the reader and remain subject to
  editorial confirmation. The reversed equality is not used here:
  the conjugation and derivative are derived directly.
- [NIST/SEMATECH, 2.3.6.1: calibration](https://www.itl.nist.gov/div898/handbook/mpc/section3/mpc361.htm),
  “Power model treated as a linear model”: multiplicative error and
  linearisation. The dimensionless references and data are part of this exposition.
- [NIST/SEMATECH, 1.3.6.6.9: lognormal distribution](https://www.itl.nist.gov/div898/handbook/eda/section3/eda3669.htm),
  parameterisation and “Common Statistics”: mean and median. The conditional
  formula is applied under the stated assumptions.
- [Python: `math`](https://docs.python.org/3.14/library/math.html),
  `log`, `exp`, `fsum` and `isclose`: implementation operations.

Those sections were consulted during preparation. The derivations, examples
and code are educational material developed for this laboratory. No scientific
priority is claimed, and the book is not distributed. The checks cover
the cited passages and examples only.


In [13]:
summary = {
    "status": "passed",
    "application": "power_laws_and_elasticity",
    "python": platform.python_version(),
    "A_noiseless": A_estimated,
    "b_noiseless": b_estimated,
    "A_fixed_errors": math.exp(a_demo),
    "b_fixed_errors": b_demo,
    "original_criterion": original_fit,
    "log_criterion": log_fit,
    "negative_domain_rejected": True,
    "empirical_model_validated": False,
    }
(RESULTS / "checks.json").write_text(
        json.dumps(summary, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
        )
print("Checks passed. CSV files and summary in:", RESULTS)

Checks passed. CSV files and summary in: results/power_laws_and_elasticity
